# 07 · Fine-Tuning, PEFT & LoRA

> **Source notes:** `FineTuning.md`

When is fine-tuning the right call? This notebook covers:
- **Decision tree** — should you fine-tune at all?
- **LoRA math visualised** — low-rank decomposition and parameter savings
- **Dataset preparation** — building and validating a fine-tuning dataset
- **PEFT config** — ready-to-adapt LoRA training setup
- **Distillation workflow** — generating synthetic data from a larger model

> GPU recommended for actual training. Math and dataset cells run on CPU.

In [ ]:
# TODO: Implement this cell


## 1 · The Fine-Tuning Decision Tree

```
Failing output format?       -> Prompt engineering / structured output
Missing domain facts?         -> RAG (fine-tuning memorises facts poorly)
Style/behaviour failure?      -> Fine-tuning v
Too slow/expensive?           -> Distillation v
Proprietary syntax/DSL?       -> Fine-tuning v
Reasoning failures?           -> Better model or CoT
```

In [ ]:
# TODO: Implement this cell


## 2 · LoRA Math — Low-Rank Decomposition

LoRA adds: **W' = W + BA** where A in R(r x k), B in R(d x r), r << min(d,k).
Trains r(d+k) params instead of d*k — typically 0.1-3% of full params.

In [ ]:
# TODO: Implement this cell


## 3 · Dataset Preparation & Validation

In [ ]:
# TODO: Implement this cell


## 4 · PEFT / LoRA Config

Key hyperparameters:
- `r=16` — rank (default; lower=fewer params, higher=more capacity)
- `lora_alpha=32` — effective scale = alpha/r = 2.0
- `target_modules` — which matrices to adapt (Q and V projections for attention)
- `lora_dropout=0.05` — regularisation

Actual training requires GPU with ~16 GB VRAM for a 7B model.

In [ ]:
# TODO: Implement this cell


## 5 · Distillation — Synthetic Training Data from Teacher

In [ ]:
# TODO: Implement this cell


## 6 · DPO — Direct Preference Optimisation

After LoRA fine-tunes *what* the model says, DPO fine-tunes *what* the model **prefers**.

### Training Data Format

```python
{
    "prompt":   "What pizza do you recommend?",
    "chosen":   "Oh, you've gotta try the Margherita — Nonna's recipe, flying out the door!",  # y_w
    "rejected": "The Margherita pizza is available."                                           # y_l
}
```

### DPO Loss

$$\mathcal{L}_{DPO} = -\mathbb{E}\!\left[\log\sigma\!\left(\beta\log\frac{\pi_\theta(y_w|x)}{\pi_{ref}(y_w|x)} - \beta\log\frac{\pi_\theta(y_l|x)}{\pi_{ref}(y_l|x)}\right)\right]$$

| Symbol | Meaning |
|---|---|
| $\pi_\theta$ | Trainable policy (your LoRA model, being updated) |
| $\pi_{ref}$ | Frozen reference (same model before DPO — never updated) |
| $\beta$ | Temperature: controls KL regularisation (start at 0.1) |
| $y_w / y_l$ | Preferred (warm voice) / rejected (flat response) |

### When to Use

| Signal | Action |
|---|---|
| Brand voice 95%+ but 5% still cold/generic | ✅ Add DPO on top of LoRA |
| Style still wrong (flat, incorrect format) | ❌ Fix LoRA first — DPO refines, doesn't correct |
| No preference pairs available | ❌ LoRA/SFT only — DPO requires (chosen, rejected) |

**PizzaBot impact:** LoRA → 95% warm responses; LoRA + DPO → 99%+ → AOV: $41.00 → $42.50 (+3.7%)

In [ ]:
# TODO: Implement DPO training with TRL DPOTrainer
#
# Steps:
# 1. Build a small preference dataset (at least 5 examples):
#    Each example needs: "prompt", "chosen" (warm Mamma Rosa voice), "rejected" (flat voice)
#    preference_data = [
#        {"prompt": "...", "chosen": "Oh, you've gotta try...", "rejected": "The pizza is available."},
#        ...
#    ]
#
# 2. Configure DPO:
#    from trl import DPOTrainer, DPOConfig
#    dpo_config = DPOConfig(
#        beta=0.1,           # KL regularisation strength
#        learning_rate=5e-7, # Lower than LoRA
#        num_train_epochs=1, # Typically 1 epoch for DPO
#        output_dir="./dpo-pizzabot",
#    )
#
# 3. Initialize DPOTrainer:
#    dpo_trainer = DPOTrainer(
#        model=lora_model,    # Your LoRA-tuned policy (will be updated)
#        ref_model=ref_model, # Frozen reference model (NOT updated)
#        args=dpo_config,
#        train_dataset=preference_dataset,
#        tokenizer=tokenizer,
#    )
#
# 4. Train and observe metrics:
#    dpo_trainer.train()
#    # Watch: rewards/chosen ↑, rewards/rejected ↓, rewards/margins ↑
#
# 5. Print final metrics and compare brand voice score before vs after DPO

raise NotImplementedError("Implement DPO training — see ch10 §5.5 in fine-tuning.md")

## Summary

| Concept | Key Takeaway |
|---|---|
| Decision tree | Most problems → prompting or RAG first; fine-tune last |
| LoRA math | r(d+k) params vs d*k full — ~1% at r=16 |
| Data > rank | Data quality beats hyperparameter tuning |
| Distillation | Teacher-generated data is most scalable path |
| DPO | Preference triples (x, y_w, y_l) align what the model prefers — no reward model needed |
| Eval gate | Run evaluation metrics before/after to measure actual gain |

**Next:** [SafetyAndHallucination/notebook.ipynb](../SafetyAndHallucination/notebook.ipynb)